In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import cv2
import os
from tqdm.notebook import tqdm
import pandas as pd
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

In [ ]:
class CVAE(tf.keras.Model):
    """Convoutional Variational Autoencoder with input shape (256, 256, 1)."""

    def __init__(self, latent_dim):
        super(CVAE, self).__init__()   

        self.latent_dim = latent_dim
        self.encoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(256, 256, 1)),
            tf.keras.layers.Conv2D(32, 3, strides=2, activation='relu'),
            tf.keras.layers.Conv2D(64, 3, strides=2, activation='relu'),
            tf.keras.layers.Conv2D(128, 3, strides=2, activation='relu'),
            tf.keras.layers.Conv2D(256, 3, strides=2, activation='relu'),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(latent_dim + latent_dim),
        ])

        self.decoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(latent_dim,)),
            tf.keras.layers.Dense(16 * 16 * 256, activation='relu'),
            tf.keras.layers.Reshape((16, 16, 256)),
            tf.keras.layers.Conv2DTranspose(256, 4, strides=2, padding='same', activation='relu'),
            tf.keras.layers.Conv2DTranspose(128, 4, strides=2, padding='same', activation='relu'),
            tf.keras.layers.Conv2DTranspose(64, 4, strides=2, padding='same', activation='relu'),
            tf.keras.layers.Conv2DTranspose(1, 4, strides=2, padding='same'),
        ])

    @tf.function
    def sample(self, eps=None):
        if eps is None:
            eps = tf.random.normal(shape=(100, self.latent_dim))
        return self.decode(eps, apply_sigmoid=True)

    def encode(self, x):
        mean, logvar = tf.split(self.encoder(x), num_or_size_splits=2, axis=1)
        return mean, logvar

    def reparameterize(self, mean, logvar):
        eps = tf.random.normal(shape=mean.shape)
        return eps * tf.exp(logvar * .5) + mean

    def decode(self, z, apply_sigmoid=False):
        logits = self.decoder(z)
        if apply_sigmoid:
            probs = tf.sigmoid(logits)
            return probs
        return logits

    def call(self, x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        reconstructed = self.decode(z, apply_sigmoid=True)
        return reconstructed

# Helper functions from your code
def get_bounding_box(ground_truth_map):
    # get bounding box from mask
    y_indices, x_indices = np.where(ground_truth_map > 0)

    if len(x_indices) == 0 or len(y_indices) == 0:
        return [0,0,255,255]

    x_min, x_max = np.min(x_indices), np.max(x_indices)
    y_min, y_max = np.min(y_indices), np.max(y_indices)
    # add perturbation to bounding box coordinates
    H, W = ground_truth_map.shape
    x_min = max(0, x_min - 5)
    x_max = min(W, x_max + 5)
    y_min = max(0, y_min - 5)
    y_max = min(H, y_max + 5)
    bbox = [x_min, y_min, x_max, y_max]

    return bbox

def preprocess_image(image):
    
    bbox = get_bounding_box(image)
    image = image[bbox[1]:bbox[3], bbox[0]:bbox[2]]
    image = cv2.resize(image, (256, 256))

    filtered_image = np.array(image)
    filtered_image = filtered_image.reshape((256, 256, 1)) 

    return filtered_image

def load_and_preprocess_image(file_path):
    """
    Load and preprocess a single image file
    """
    img = tifffile.imread(file_path)
    img = img.astype(np.float32) / 4000.0  # Normalize based on your dataset
    processed_img = preprocess_image(img)
    
    return processed_img

In [ ]:
class VAEGradCAM:
    def __init__(self, model, latent_dim=16):
        self.model = model
        self.latent_dim = latent_dim
        
        # Get the encoder
        self.encoder = model.encoder
        
        # Target layer is the last conv layer
        self.target_layer = None
        for layer in self.encoder.layers:
            if isinstance(layer, tf.keras.layers.Conv2D):
                self.target_layer = layer
                
    def compute_heatmap(self, input_image, latent_dim_idx, layer_name=None):
        if not isinstance(input_image, tf.Tensor):
            input_image = tf.convert_to_tensor(input_image, dtype=tf.float32)
            
        if len(input_image.shape) == 3:
            input_image = tf.expand_dims(input_image, axis=0)
            
        # Create a model that maps the input to the target layer activations
        # and the output of the encoder
        grad_model = tf.keras.models.Model(
            inputs=[self.encoder.inputs],
            outputs=[self.target_layer.output, self.encoder.output]
        )
            
        # Compute gradients
        with tf.GradientTape() as tape:
            # Get target layer output and encoder output
            conv_output, encoder_output = grad_model(input_image)
            
            # Split encoder output into mean and logvar
            mean, _ = tf.split(encoder_output, num_or_size_splits=2, axis=1)
            
            # Get the specific latent dimension
            target_output = mean[:, latent_dim_idx]
            
        # Compute gradients with respect to the target layer
        grads = tape.gradient(target_output, conv_output)
        
        # Global average pooling
        pooled_grads = tf.reduce_mean(grads, axis=(1, 2))
        
        # Weight the channels by their average gradients
        heatmap = tf.matmul(conv_output[0], pooled_grads[..., tf.newaxis])
        heatmap = tf.squeeze(heatmap)
        
        # ReLU the heatmap
        heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + tf.keras.backend.epsilon())
        
        # Convert to numpy
        heatmap = heatmap.numpy()
        
        # Resize to input dimensions
        heatmap = cv2.resize(heatmap, (input_image.shape[2], input_image.shape[1]))
        
        return heatmap
        
    def overlay_heatmap(self, heatmap, image, alpha=0.7):
        """Overlay heatmap on original image"""
        # Make a copy
        img = np.squeeze(image.copy())
        
        # Normalize image if needed
        if img.max() > 1.0:
            img = img / img.max()
            
        # Convert to RGB for overlay
        img_rgb = np.repeat(img[..., np.newaxis], 3, axis=2)
        
        # Create heatmap
        colored_heatmap = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
        colored_heatmap = cv2.cvtColor(colored_heatmap, cv2.COLOR_BGR2RGB)
        
        # Overlay
        overlay = (colored_heatmap * alpha + img_rgb * (1 - alpha)).astype(np.uint8)
        
        return overlay
        
    def generate_all_heatmaps(self, input_image):
        """Generate heatmaps for all latent dimensions"""
        heatmaps = []
        
        for i in range(self.latent_dim):
            heatmap = self.compute_heatmap(input_image, i)
            heatmaps.append(heatmap)
            
        return heatmaps
        
    def visualize_heatmaps(self, input_image, save_path=None):
        """Visualize heatmaps for all latent dimensions"""
        heatmaps = self.generate_all_heatmaps(input_image)
        
        # Calculate grid size
        grid_size = int(np.ceil(np.sqrt(self.latent_dim + 1)))
        
        # Create figure
        fig, axs = plt.subplots(grid_size, grid_size, figsize=(15, 15))
        axs = axs.flatten()
        
        # Plot original image
        axs[0].imshow(np.squeeze(input_image), cmap='gray')
        axs[0].set_title('Original Image')
        axs[0].axis('off')
        
        # Plot heatmaps
        for i, heatmap in enumerate(heatmaps):
            axs[i+1].imshow(heatmap, cmap='jet')
            axs[i+1].set_title(f'Latent Dim {i}')
            axs[i+1].axis('off')
            
        # Remove empty subplots
        for i in range(self.latent_dim + 1, grid_size * grid_size):
            fig.delaxes(axs[i])
            
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            
        plt.show()
        
        return heatmaps
    
    def visualize_single_latent(self, input_image, latent_dim_idx, save_path=None):
        """Visualize heatmap for a single latent dimension with overlay"""
        heatmap = self.compute_heatmap(input_image, latent_dim_idx)
        overlay = self.overlay_heatmap(heatmap, input_image)
        
        # Display
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 3, 1)
        plt.imshow(np.squeeze(input_image), cmap='gray')
        plt.title('Original Image')
        plt.axis('off')
        
        plt.subplot(1, 3, 2)
        plt.imshow(heatmap, cmap='jet')
        plt.title(f'Attention Map (Latent {latent_dim_idx})')
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        plt.imshow(overlay)
        plt.title('Overlay')
        plt.axis('off')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            
        plt.show()
        
        return heatmap, overlay

In [ ]:
def visualize_latent_dimension_influence(model, input_image, save_dir=None):
    """Visualize how each latent dimension influences reconstruction"""
    if not isinstance(input_image, tf.Tensor):
        input_image = tf.convert_to_tensor(input_image, dtype=tf.float32)
        
    if len(input_image.shape) == 3:
        input_image = tf.expand_dims(input_image, axis=0)
        
    # Get mean and logvar
    mean, logvar = model.encode(input_image)
    
    # Get original reconstruction
    z = model.reparameterize(mean, logvar)
    original_reconstruction = model.decode(z, apply_sigmoid=True)
    
    # For each latent dimension, zero it out and observe effect
    altered_reconstructions = []
    
    for i in range(model.latent_dim):
        # Make a copy
        z_altered = tf.identity(z)
        
        # Zero out dimension i
        z_altered = tf.tensor_scatter_nd_update(
            z_altered, 
            [[0, i]], 
            [tf.zeros(1)]
        )
        
        # Get reconstruction
        altered_reconstruction = model.decode(z_altered, apply_sigmoid=True)
        altered_reconstructions.append(altered_reconstruction)
        
    # Visualize
    num_cols = 4
    num_rows = int(np.ceil((model.latent_dim + 2) / num_cols))
    
    fig, axs = plt.subplots(num_rows, num_cols, figsize=(15, num_rows * 4))
    axs = axs.flatten()
    
    # Plot original image
    axs[0].imshow(np.squeeze(input_image), cmap='gray')
    axs[0].set_title('Original Image')
    axs[0].axis('off')
    
    # Plot original reconstruction
    axs[1].imshow(np.squeeze(original_reconstruction), cmap='gray')
    axs[1].set_title('Full Reconstruction')
    axs[1].axis('off')
    
    # Plot altered reconstructions
    for i, recon in enumerate(altered_reconstructions):
        axs[i+2].imshow(np.squeeze(recon), cmap='gray')
        axs[i+2].set_title(f'Without Latent {i}')
        axs[i+2].axis('off')
        
    # Remove empty subplots
    for i in range(model.latent_dim + 2, num_rows * num_cols):
        fig.delaxes(axs[i])
        
    plt.tight_layout()
    
    if save_dir:
        plt.savefig(os.path.join(save_dir, "latent_influence.png"), dpi=300, bbox_inches='tight')
        
    plt.show()
    
    return altered_reconstructions

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
WEIGHTS_PATH = "./data/shriya/SHMOLLI-VAE-output/cvae_16d_optimized.weights.h5"
IMAGES_DIR = "./data/shriya/SHMOLLI-output-unet-myocardium/SAM_masks"
OUTPUT_DIR = "./attention_maps_output"  # Change as needed
LATENT_DIM = 16

In [ ]:
# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the model
model = CVAE(LATENT_DIM)
model.build(input_shape=(None, 256, 256, 1))
model.load_weights(WEIGHTS_PATH)
print(f"Model loaded from {WEIGHTS_PATH}")

In [ ]:
# Get a list of images to process
image_files = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith('.tiff')])
print(f"Found {len(image_files)} images in {IMAGES_DIR}")

# Create GradCAM instance
gradcam = VAEGradCAM(model, LATENT_DIM)

In [ ]:
# Process a few sample images
num_samples = 5  # Adjust as needed
sample_indices = np.random.choice(len(image_files), min(num_samples, len(image_files)), replace=False)

for idx in sample_indices:
    # Load and preprocess image
    image_path = os.path.join(IMAGES_DIR, image_files[idx])
    print(f"Processing {image_path}")
    
    # Load image
    img = tifffile.imread(image_path)
    img = img.astype(np.float32) / 4000.0  # Normalize
    processed_img = preprocess_image(img)
    
    # Generate attention maps for all latent dimensions
    heatmaps = gradcam.visualize_heatmaps(processed_img, 
                                        save_path=os.path.join(OUTPUT_DIR, f"{os.path.basename(image_path)}_heatmaps.png"))
    
    # Also create overlays for each latent dimension
    for latent_idx in range(LATENT_DIM):
        heatmap, overlay = gradcam.visualize_single_latent(
            processed_img, 
            latent_idx, 
            save_path=os.path.join(OUTPUT_DIR, f"{os.path.basename(image_path)}_latent{latent_idx}_overlay.png")
        )
    
    # Visualize latent dimension influence on reconstruction
    visualize_latent_dimension_influence(
        model, 
        processed_img, 
        save_dir=os.path.join(OUTPUT_DIR, f"{os.path.basename(image_path)}_latent_influence")
    )

In [ ]:
# Interactive widget for exploring images and latent dimensions
# Note: This will only work in a Jupyter notebook
try:
    from ipywidgets import interact, IntSlider, Dropdown
    
    # Load a few sample images
    sample_paths = [os.path.join(IMAGES_DIR, image_files[i]) for i in sample_indices]
    sample_images = []
    for path in sample_paths:
        img = tifffile.imread(path)
        img = img.astype(np.float32) / 4000.0
        processed_img = preprocess_image(img)
        sample_images.append(processed_img)
    
    # Create interactive widget
    def explore_attention(image_idx=0, latent_idx=0):
        image = sample_images[image_idx]
        gradcam.visualize_single_latent(image, latent_idx)
        print(f"Showing latent dimension {latent_idx} for image {os.path.basename(sample_paths[image_idx])}")
    
    interact(
        explore_attention,
        image_idx=IntSlider(min=0, max=len(sample_images)-1, step=1, value=0, description='Image:'),
        latent_idx=IntSlider(min=0, max=LATENT_DIM-1, step=1, value=0, description='Latent Dim:')
    )
except ImportError:
    print("Interactive widgets require ipywidgets to be installed.")
    
print("Attention map generation complete!")